In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd, numpy as np, os
from google.colab import drive
drive.mount('/content/drive')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
DATA_DIR = "/content/drive/MyDrive/IDS_Project/"

cols = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
"wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
"root_shell","su_attempted","num_root","num_file_creations","num_shells",
"num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
"srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
"same_srv_rate","diff_srv_rate","srv_diff_srv_rate","dst_host_count",
"dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
"dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
"dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
"label","difficulty"]

print(os.listdir(DATA_DIR + "data"))

train = pd.read_csv(DATA_DIR + "data/KDDTrain+.txt", names=cols)
test  = pd.read_csv(DATA_DIR + "data/KDDTest+.txt",  names=cols)

print("Train:", train.shape, " Test:", test.shape)
print("Total rows:", len(train) + len(test))
print("Missing values (train/test):", train.isnull().sum().sum(), test.isnull().sum().sum())
print("Duplicates (train/test):", train.duplicated().sum(), test.duplicated().sum())
print("Unique attack labels in train:", train['label'].nunique())
print(train['label'].value_counts().head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['KDDTest+.txt', 'KDDTrain+.txt']
Train: (125973, 43)  Test: (22544, 43)
Total rows: 148517
Missing values (train/test): 0 0
Duplicates (train/test): 0 0
Unique attack labels in train: 23
label
normal         67343
neptune        41214
satan           3633
ipsweep         3599
portsweep       2931
smurf           2646
nmap            1493
back             956
teardrop         892
warezclient      890
Name: count, dtype: int64


In [3]:
import pandas as pd, numpy as np, os

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
DATA_DIR = "/content/drive/MyDrive/IDS_Project/"

print(os.listdir(DATA_DIR + "data"))   # KDDTest+.txt, KDDTrain+.txt

train = pd.read_csv(DATA_DIR + "data/KDDTrain+.txt", names=cols)
test  = pd.read_csv(DATA_DIR + "data/KDDTest+.txt",  names=cols)

print("Train:", train.shape, " Test:", test.shape)
print("Missing values (train/test):", train.isnull().sum().sum(), test.isnull().sum().sum())
print("Duplicates (train/test):", train.duplicated().sum(), test.duplicated().sum())

['KDDTest+.txt', 'KDDTrain+.txt']
Train: (125973, 43)  Test: (22544, 43)
Missing values (train/test): 0 0
Duplicates (train/test): 0 0


In [4]:
import os, joblib
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
DATA_DIR = "/content/drive/MyDrive/IDS_Project/"

cols = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
"wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
"root_shell","su_attempted","num_root","num_file_creations","num_shells",
"num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
"srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
"same_srv_rate","diff_srv_rate","srv_diff_srv_rate","dst_host_count",
"dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
"dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
"dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
"label","difficulty"]

train = pd.read_csv(DATA_DIR + "data/KDDTrain+.txt", names=cols)
test  = pd.read_csv(DATA_DIR + "data/KDDTest+.txt",  names=cols)
print(train.shape, test.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(125973, 43) (22544, 43)


In [5]:
attack_map = {'normal': 'Normal'}
for a in ['back','land','neptune','pod','smurf','teardrop','apache2','udpstorm',
          'processtable','mailbomb','worm']: attack_map[a] = 'DoS'
for a in ['satan','ipsweep','nmap','portsweep','mscan','saint']: attack_map[a] = 'Probe'
for a in ['guess_passwd','ftp_write','imap','phf','multihop','warezmaster','warezclient',
          'spy','xlock','xsnoop','snmpguess','snmpgetattack','httptunnel','sendmail','named']:
    attack_map[a] = 'R2L'
for a in ['buffer_overflow','loadmodule','rootkit','perl','sqlattack','xterm','ps']:
    attack_map[a] = 'U2R'

class_names = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']   # index = integer label
class_to_int = {c: i for i, c in enumerate(class_names)}

for df in (train, test):
    df['attack_class'] = df['label'].map(attack_map)

# VERIFY mapping
print("Unmapped labels (train/test):", train['attack_class'].isna().sum(), test['attack_class'].isna().sum())
print("Raw labels not in map:", set(train['label']).union(test['label']) - set(attack_map))
print("Classes found:", sorted(set(train['attack_class']) | set(test['attack_class'])))
assert set(train['attack_class']) | set(test['attack_class']) == set(class_names)

y_train = train['attack_class'].map(class_to_int).values.astype(int)
y_test  = test['attack_class'].map(class_to_int).values.astype(int)

Unmapped labels (train/test): 0 0
Raw labels not in map: set()
Classes found: ['DoS', 'Normal', 'Probe', 'R2L', 'U2R']


In [6]:
dist = pd.DataFrame({
    'train_count': pd.Series(y_train).map(dict(enumerate(class_names))).value_counts(),
    'test_count':  pd.Series(y_test).map(dict(enumerate(class_names))).value_counts()
}).reindex(class_names)
dist['train_%'] = (dist['train_count'] / dist['train_count'].sum() * 100).round(2)
dist['test_%']  = (dist['test_count']  / dist['test_count'].sum()  * 100).round(2)
print(dist)

        train_count  test_count  train_%  test_%
Normal        67343        9711    53.46   43.08
DoS           45927        7460    36.46   33.09
Probe         11656        2421     9.25   10.74
R2L             995        2885     0.79   12.80
U2R              52          67     0.04    0.30


In [7]:
from sklearn.preprocessing import OneHotEncoder, RobustScaler

X_tr = train.drop(columns=['label', 'difficulty', 'attack_class'])
X_te = test.drop(columns=['label', 'difficulty', 'attack_class'])

cat_cols    = ['protocol_type', 'service', 'flag']
binary_cols = ['land', 'logged_in', 'root_shell', 'su_attempted', 'is_host_login', 'is_guest_login']
num_cols    = [c for c in X_tr.columns if c not in cat_cols + binary_cols]

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
scaler = RobustScaler()
ohe.fit(X_tr[cat_cols])          # TRAIN only
scaler.fit(X_tr[num_cols])       # TRAIN only

def transform(X):
    return np.hstack([scaler.transform(X[num_cols]),
                      X[binary_cols].values,
                      ohe.transform(X[cat_cols])]).astype(np.float32)

X_train, X_test = transform(X_tr), transform(X_te)
feature_names = num_cols + binary_cols + list(ohe.get_feature_names_out(cat_cols))

# VERIFY
assert X_train.shape[1] == X_test.shape[1] == len(feature_names)
assert not np.isnan(X_train).any() and not np.isnan(X_test).any()
print("Scaler fitted on", scaler.n_samples_seen_ if hasattr(scaler,'n_samples_seen_') else len(X_tr), "rows (train size =", len(X_tr), ")")
print("OHE categories fitted from train only. Unseen test categories:",
      {c: sorted(set(X_te[c]) - set(X_tr[c])) for c in cat_cols})
print(X_train.shape, X_test.shape, len(feature_names))

Scaler fitted on 125973 rows (train size = 125973 )
OHE categories fitted from train only. Unseen test categories: {'protocol_type': [], 'service': [], 'flag': []}
(125973, 122) (22544, 122) 122


In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report)

results = {}
models = {}
for crit in ['gini', 'entropy']:
    dt = DecisionTreeClassifier(criterion=crit, random_state=RANDOM_STATE)
    dt.fit(X_train, y_train)
    pred = dt.predict(X_test)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, average='macro', zero_division=0)
    results[crit] = {'Accuracy': accuracy_score(y_test, pred),
                     'Macro Precision': p, 'Macro Recall': r, 'Macro F1': f}
    models[crit] = dt
    print(f"\n===== Decision Tree ({crit}) | depth={dt.get_depth()} leaves={dt.get_n_leaves()} =====")
    print(classification_report(y_test, pred, labels=range(5),
                                target_names=class_names, digits=4, zero_division=0))

print(pd.DataFrame(results).T.round(4))


===== Decision Tree (gini) | depth=28 leaves=317 =====
              precision    recall  f1-score   support

      Normal     0.6629    0.9622    0.7850      9711
         DoS     0.9565    0.8143    0.8797      7460
       Probe     0.7916    0.6212    0.6961      2421
         R2L     0.9593    0.0572    0.1079      2885
         U2R     0.5600    0.2090    0.3043        67

    accuracy                         0.7586     22544
   macro avg     0.7861    0.5328    0.5546     22544
weighted avg     0.8115    0.7586    0.7187     22544


===== Decision Tree (entropy) | depth=17 leaves=304 =====
              precision    recall  f1-score   support

      Normal     0.6867    0.9707    0.8043      9711
         DoS     0.9638    0.8005    0.8746      7460
       Probe     0.7819    0.7584    0.7700      2421
         R2L     0.9563    0.0835    0.1536      2885
         U2R     0.4286    0.1343    0.2045        67

    accuracy                         0.7756     22544
   macro avg    

In [9]:
OUT = DATA_DIR + "output/"
os.makedirs(OUT, exist_ok=True)

processed_data = {
    "X_train": X_train, "X_test": X_test,
    "y_train": y_train, "y_test": y_test,
    "class_names": class_names,
    "feature_names": feature_names
}
joblib.dump(processed_data, "/content/drive/MyDrive/IDS_Project/output/processed.pkl", compress=3)

['/content/drive/MyDrive/IDS_Project/output/processed.pkl']

In [10]:
import joblib
d = joblib.load("/content/drive/MyDrive/IDS_Project/output/processed.pkl")
print("X_train:", d["X_train"].shape)
print("X_test: ", d["X_test"].shape)
print("y_train:", d["y_train"].shape)
print("y_test: ", d["y_test"].shape)
print("class_names:", d["class_names"])
print("n feature_names:", len(d["feature_names"]))
assert d["X_train"].shape[1] == len(d["feature_names"])

X_train: (125973, 122)
X_test:  (22544, 122)
y_train: (125973,)
y_test:  (22544,)
class_names: ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
n feature_names: 122
